In [16]:
import torch
from torch import nn
import nltk
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader
import json
import pandas as pd

In [3]:
class Tokenizer:
    def __init__(self, language: str):
        self.language = language

        self.special_token = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]

        self.id2token = []
        self.token2id = {}

    # 分词
    def tokenize(self, sentences: list) -> list:
        tokens = []

        if self.language == "zh":
            for text in sentences:
                tokens.append(list(text))

        elif self.language == "en":
            for text in sentences:
                tokens.append(word_tokenize(text))

        
        return tokens

    # 构造词表
    def create_vocab(self, tokens: list) -> None:
        self.id2token += self.special_token

        for sentence in tokens:
            for token in sentence:
                if token not in self.id2token:
                    self.id2token.append(token)

        for index, token in enumerate(self.id2token):
            self.token2id[token] = index

    # padding
    def padding(self, ids_list: list) -> list:
        max_len = max(len(ids) for ids in ids_list)

        res = []

        for ids in ids_list:
            ids += [self.token2id["<PAD>"]] * (max_len - len(ids))

            res.append(ids)

        return res

    # 编码
    def encode(self, text: list) -> list:
        tokens = self.tokenize(text)

        res = []

        for sentence in tokens:
            sentence = ["<SOS>"] + sentence + ["<EOS>"]

            # 编码成id列表
            ids = [self.token2id.get(token, self.token2id["<UNK>"]) for token in sentence]

            res.append(ids)

        res = self.padding(res)

        return res

    # 解码
    def decode(self, ids: list) -> list:
        res = []

        for id_list in ids:
            tokens = [self.id2token[token_id] for token_id in id_list]

            tokens = [token for token in tokens if token not in self.special_token]

            if self.language == "en":
                tokens = " ".join(tokens)
            elif self.language == "zh":
                tokens = "".join(tokens)

            res.append(tokens)

        return res

In [ ]:
data = [
    ("我爱你", "I love you"),
    ("你好", "hello"),
    ("我喜欢猫", "I like cats"),
    ("我喜欢狗", "I like dogs"),
    ("你爱猫", "you love cats"),
]

src_sentences = [item[0] for item in data]
tgt_sentences = [item[1] for item in data]

src_tokenizer = Tokenizer("zh")
tgt_tokenizer = Tokenizer("en")

src_tokens = src_tokenizer.tokenize(src_sentences)
src_tokenizer.create_vocab(src_tokens)
print(src_tokenizer.token2id)
tgt_tokens = tgt_tokenizer.tokenize(tgt_sentences)
tgt_tokenizer.create_vocab(tgt_tokens)
print(tgt_tokenizer.token2id)

src_ids = src_tokenizer.encode(src_sentences)
tgt_ids = tgt_tokenizer.encode(tgt_sentences)

{'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3, '我': 4, '爱': 5, '你': 6, '好': 7, '喜': 8, '欢': 9, '猫': 10, '狗': 11}
{'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3, 'I': 4, 'love': 5, 'you': 6, 'hello': 7, 'like': 8, 'cats': 9, 'dogs': 10}
[[2, 4, 5, 6, 3, 0], [2, 6, 7, 3, 0, 0], [2, 4, 8, 9, 10, 3], [2, 4, 8, 9, 11, 3], [2, 6, 5, 10, 3, 0]]
[[2, 4, 5, 6, 3], [2, 7, 3, 0, 0], [2, 4, 8, 9, 3], [2, 4, 8, 10, 3], [2, 6, 5, 9, 3]]


In [ ]:
with open("index_train.jsonl", "w", encoding="utf-8") as f:
    data = []

    for src_id, tgt_id in zip(src_ids, tgt_ids):
        data = {"src": src_id, "tgt": tgt_id}

        f.write(json.dumps(data, ensure_ascii=False) + '\n')

In [22]:
class TranslationDataset(Dataset):
    def __init__(self, data_path, src_tokenizer, tgt_tokenizer):
        super().__init__()

        self.data = pd.read_json(data_path, lines=True).to_dict(orient="records")
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer

    # 返回数据的长度
    def __len__(self):
        return len(self.data)

    # 获取指定的数据
    def __getitem__(self, index):
        src_tensor = torch.tensor(self.data[index]["src"])
        tgt_tensor = torch.tensor(self.data[index]["tgt"])

        return src_tensor, tgt_tensor

In [23]:
dataset = TranslationDataset("index_train.jsonl", src_tokenizer, tgt_tokenizer)

print(dataset[0])

(tensor([2, 4, 5, 6, 3, 0]), tensor([2, 4, 5, 6, 3]))


In [26]:
def create_dataloader():
    dataset = TranslationDataset("index_train.jsonl", src_tokenizer, tgt_tokenizer)

    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    return dataloader

dataloader = create_dataloader()

In [28]:
for src, tgt in dataloader:
    print(src)
    print(tgt)

print(len(dataloader))

tensor([[ 2,  6,  5, 10,  3,  0],
        [ 2,  4,  5,  6,  3,  0],
        [ 2,  6,  7,  3,  0,  0],
        [ 2,  4,  8,  9, 10,  3],
        [ 2,  4,  8,  9, 11,  3]])
tensor([[ 2,  6,  5,  9,  3],
        [ 2,  4,  5,  6,  3],
        [ 2,  7,  3,  0,  0],
        [ 2,  4,  8,  9,  3],
        [ 2,  4,  8, 10,  3]])
1
